# Prefix Surgery: Full Workflow

This notebook runs the complete prefix-surgery experiment and its clean-rewrite
control for the NVIDIA Nemotron Model Reasoning Challenge.

**Workflow:**

1. **Baseline adapter** - download and validate
   [Kien's public rank-32 adapter](https://www.kaggle.com/models/kienngx/nemotron-nano-30b-trained).
2. **Baseline generation** - run the adapter over all 9,500 rows and save its
   raw reasoning traces.
3. **Solver report** - solve the `equation_symbol_cipher` rows with the
   deterministic Python solver.
4. **Corpus build** - create Arm A prefix-surgery traces, Arm B clean-rewrite
   traces, shared replay anchors, and verifier reports.
5. **Training** - train both arms concurrently on Modal.
6. **Evaluation** - evaluate both arms on the full dataset and show the
   canonical held-out comparison by task family and variant.

The code cells persist all cross-stage artifacts on a Modal Volume, so completed
stages can be reused when the notebook configuration allows it.

## Configuration

Every value below is the exact setting used for the reported run. The only
things you normally change are `VOLUME_NAME` and `LOCAL_TRAIN_CSV`.
`LOCAL_TRAIN_CSV` (path to your Kaggle `train.csv`).

In [1]:
import uuid
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the checkout regardless of where Jupyter was launched."""
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / "nemotron-reasoning-challenge-prefix-surgery"):
            if (
                (candidate / "pyproject.toml").is_file()
                and (candidate / "nemotron_reasoning").is_dir()
                and (candidate / "notebooks" / "full_workflow.ipynb").is_file()
            ):
                return candidate
    raise RuntimeError("Could not find the repository root from the current working directory")

# --- Your environment -------------------------------------------------------
VOLUME_NAME = "nemotron-prefix-surgery"         # Modal volume to create/use
TRAIN_GPU = "H200:2"                            # two-process DDP per arm
EVAL_GPU = "H200"                               # one GPU per inference job
MODAL_TIMEOUT_SECONDS = 24 * 60 * 60
REUSE_EXISTING_RUNS = False
WORKFLOW_SESSION_ID = uuid.uuid4().hex

REPO_ROOT = find_repo_root()
LOCAL_TRAIN_CSV = REPO_ROOT / "data" / "train.csv"
WORK = REPO_ROOT / "_repro_work"
WORK.mkdir(parents=True, exist_ok=True)

# --- Fixed identifiers (match the reported run) -----------------------------
BASE_MODEL = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
KIEN_MODEL_HANDLE = "kienngx/nemotron-nano-30b-trained/Triton/tinker-adapter/1"
BASELINE_RUN = "kien_tinker_086_baseline"       # warm-start adapter + baseline preds live here
SURGERY_RUN = "eqsym_surgery_001"               # solver report + corpora live here
ARM_A_RUN = "eqsym_surgery_A_001"
ARM_B_RUN = "eqsym_surgery_B_001"

# --- Solver (Stage 3) -------------------------------------------------------
SOLVER_WORKERS = 12
SOLVER_ROW_BUDGET = 60.0
SOLVER_UNIQUENESS_BUDGET = 5.0

# --- Corpus build (Stage 4) -------------------------------------------------
CORPUS_SEED = 42
ANCHOR_PER_FAMILY = 300

# --- Training (Stage 5) — the reported Arm A/B config -----------------------
SEED = 12345
MAX_STEPS = 223
MAX_SEQ_LENGTH = 8192
PER_DEVICE_BATCH = 1
TRAIN_WORLD_SIZE = 2
GRAD_ACCUM = 8
EFFECTIVE_BATCH = PER_DEVICE_BATCH * GRAD_ACCUM * TRAIN_WORLD_SIZE
LEARNING_RATE = 2e-5
LR_SCHEDULER = "cosine"
WARMUP_STEPS = 7
MAX_GRAD_NORM = 1.0
CHECKPOINT_FRACTION = 0.25
CHECKPOINT_STEPS = round(MAX_STEPS * CHECKPOINT_FRACTION)

# --- Evaluation (Stage 6) — the official metric settings --------------------
EVAL_MAX_TOKENS = 7680
EVAL_MAX_NUM_SEQS = 64
VALID_FRACTION = 0.2          # canonical held-out split
SPLIT_SEED = 12345
EXPECTED_FULL_ROWS = 9500
EXPECTED_VALID_ROWS = 1899

if EFFECTIVE_BATCH != 16:
    raise AssertionError(f"The controlled experiment requires global batch 16, got {EFFECTIVE_BATCH}")

# Make the local package importable when running from notebooks/.
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## Modal app

The app, image, and GPU/CPU functions are defined here so the repo needs no
separate `modal_app.py`. The image mirrors the one used for the reported run
(vLLM + TRL/PEFT + the Mamba kernels Nemotron-H needs). The local
`nemotron_reasoning` package is baked into the image; each function imports it
at call time and reads/writes the mounted volume.

In [2]:
import json
import os
import subprocess
import sys
import time

import modal

MOUNT_ROOT = "/mnt/nemotron-prefix-surgery"

app = modal.App("prefix-surgery-reproduction")
volume = modal.Volume.from_name(VOLUME_NAME, create_if_missing=True)

core_packages = [
    "accelerate==1.13.0",
    "bitsandbytes==0.49.2",
    "datasets==4.8.5",
    "ninja==1.13.0",
    "numpy==2.3.5",
    "packaging==26.2",
    "pandas==3.0.3",
    "peft==0.19.1",
    "pyarrow==24.0.0",
    "safetensors==0.7.0",
    "setuptools==81.0.0",
    "torch==2.11.0",
    "transformers==5.9.0",
    "trl==1.5.1",
    "vllm==0.22.0",
    "wheel==0.47.0",
]
kernel_packages = [
    "causal-conv1d==1.6.2.post1",
    "mamba-ssm==2.3.2.post1",
]

image = (
    modal.Image.from_registry("nvidia/cuda:13.0.2-devel-ubuntu22.04", add_python="3.11")
    .entrypoint([])
    .apt_install("build-essential", "git")
    .env(
        {
            "CUDA_HOME": "/usr/local/cuda",
            "PATH": "/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin",
            "CC": "/usr/bin/gcc",
            "CXX": "/usr/bin/g++",
            "CUDAHOSTCXX": "/usr/bin/g++",
        }
    )
    .uv_pip_install(*core_packages)
    .uv_pip_install(*kernel_packages, extra_options="--no-build-isolation")
    .env(
        {
            "PYTHONPATH": "/root",
            "HF_HOME": f"{MOUNT_ROOT}/hf-cache",
            "HF_HUB_CACHE": f"{MOUNT_ROOT}/hf-cache/hub",
            "HF_DATASETS_CACHE": f"{MOUNT_ROOT}/hf-cache/datasets",
            "TOKENIZERS_PARALLELISM": "false",
        }
    )
    .add_local_dir(str(REPO_ROOT / "nemotron_reasoning"), remote_path="/root/nemotron_reasoning")
)

mounted = {"volumes": {MOUNT_ROOT: volume}, "image": image}


def validate_adapter_dir(adapter_dir: Path, expected_rank: int = 32) -> dict:
    config_path = adapter_dir / "adapter_config.json"
    weights_path = adapter_dir / "adapter_model.safetensors"
    if not config_path.is_file() or not weights_path.is_file():
        raise FileNotFoundError(f"Incomplete adapter at {adapter_dir}")
    config = json.loads(config_path.read_text(encoding="utf-8"))
    rank = int(config.get("r", 0))
    if rank != expected_rank:
        raise ValueError(f"Expected rank {expected_rank}, found rank {rank} at {adapter_dir}")
    return {"path": str(adapter_dir), "rank": rank, "weights_bytes": weights_path.stat().st_size}


def commit_complete_checkpoints(run_id: str, committed: set[str]) -> None:
    checkpoint_root = Path(MOUNT_ROOT) / "runs" / run_id / "checkpoints"
    for marker in sorted(checkpoint_root.glob("checkpoint-*/modal_checkpoint_complete.json")):
        key = str(marker.parent)
        if key not in committed:
            volume.commit()
            committed.add(key)
            print(f"Committed complete checkpoint: {key}", flush=True)


@app.function(
    gpu=EVAL_GPU,
    timeout=MODAL_TIMEOUT_SECONDS,
    retries=modal.Retries(max_retries=3),
    single_use_containers=True,
    **mounted,
)
def baseline_or_eval_infer(
    run_id: str,
    csv_path: str,
    output_name: str,
    limit: int = 0,
    tensor_parallel_size: int = 1,
    use_adapter: bool = True,
    reuse_existing: bool = False,
) -> dict:
    """vLLM inference under the official metric settings (Stages 2 and 6)."""
    from nemotron_reasoning.inference import run_vllm_inference

    root = Path(MOUNT_ROOT) / "runs" / run_id
    prediction_path = root / "predictions" / Path(output_name).name
    summary_path = root / "eval" / f"{Path(output_name).stem}_summary.json"
    if prediction_path.exists() or summary_path.exists():
        if reuse_existing and prediction_path.is_file() and summary_path.is_file():
            print(f"Reusing completed inference artifact: {prediction_path}", flush=True)
            return json.loads(summary_path.read_text(encoding="utf-8"))
        raise FileExistsError(
            f"Inference output already exists for {run_id}: {prediction_path}. "
            "Set REUSE_EXISTING_RUNS=True only after validating the artifact."
        )

    summary = run_vllm_inference(
        run_id=run_id,
        eval_csv=Path(MOUNT_ROOT) / csv_path,
        base_model=BASE_MODEL,
        limit=limit,
        start=0,
        tensor_parallel_size=tensor_parallel_size,
        max_tokens=EVAL_MAX_TOKENS,
        max_num_seqs=EVAL_MAX_NUM_SEQS,
        output_name=output_name,
        enable_prefix_caching=False,   # Nemotron-H/Mamba prefix caching is experimental
        enable_chunked_prefill=True,
        use_adapter=use_adapter,
    )
    volume.commit()
    return summary


@app.function(timeout=MODAL_TIMEOUT_SECONDS, **mounted)
def prefilter_overlength(arm_a_csv: str, arm_b_csv: str, max_seq_length: int = MAX_SEQ_LENGTH) -> dict:
    """Drop rows whose chat-template-composed length exceeds the context budget.

    The trainer hard-asserts (never truncates) on any example longer than
    `max_seq_length`, because truncation could cut the boxed answer. The
    reported run pre-filtered the over-budget union from *both* arms so the A/B
    row sets stay identical. We measure length exactly as the trainer does.
    """
    from transformers import AutoTokenizer

    from nemotron_reasoning.io_utils import read_csv_rows, write_csv_rows
    from nemotron_reasoning.training import apply_chat_template_to_records

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

    def over_length_ids(rel_path: str) -> tuple[list[dict], set[str]]:
        rows = read_csv_rows(Path(MOUNT_ROOT) / rel_path)
        composed = apply_chat_template_to_records(rows, tokenizer)
        over: set[str] = set()
        for raw, record in zip(rows, composed):
            text = record["prompt"] + record["completion"]
            if len(tokenizer(text, add_special_tokens=False).input_ids) > max_seq_length:
                over.add(raw["id"])
        return rows, over

    a_rows, a_over = over_length_ids(arm_a_csv)
    b_rows, b_over = over_length_ids(arm_b_csv)
    drop = a_over | b_over  # union keeps the A/B id sets identical
    fields = ["id", "prompt", "prefix", "completion"]
    a_keep = [r for r in a_rows if r["id"] not in drop]
    b_keep = [r for r in b_rows if r["id"] not in drop]
    write_csv_rows(Path(MOUNT_ROOT) / arm_a_csv, a_keep, fields)
    write_csv_rows(Path(MOUNT_ROOT) / arm_b_csv, b_keep, fields)
    volume.commit()
    return {"dropped_ids": sorted(drop), "arm_a_rows": len(a_keep), "arm_b_rows": len(b_keep)}


@app.function(
    gpu=TRAIN_GPU,
    timeout=MODAL_TIMEOUT_SECONDS,
    retries=modal.Retries(max_retries=3),
    single_use_containers=True,
    **mounted,
)
def train_arm_ddp(
    run_id: str,
    train_csv: str,
    workflow_session_id: str,
    reuse_existing: bool = False,
) -> str:
    """Warm-start one arm with two-process DDP while preserving global batch 16."""
    from nemotron_reasoning.paths import run_dir

    adapter_dir = run_dir(run_id) / "adapter"
    checkpoint_root = run_dir(run_id) / "checkpoints"
    complete_checkpoints = sorted(checkpoint_root.glob("checkpoint-*/modal_checkpoint_complete.json"))
    adapter_exists = (adapter_dir / "adapter_config.json").exists()
    session_path = run_dir(run_id) / "config" / "workflow_session_id.txt"
    same_session = False
    if session_path.exists():
        existing_session = session_path.read_text(encoding="utf-8").strip()
        same_session = existing_session == workflow_session_id
        if existing_session != workflow_session_id and not reuse_existing:
            raise FileExistsError(
                f"Artifacts for {run_id} belong to another notebook session. "
                "Set REUSE_EXISTING_RUNS=True to resume them explicitly."
            )
    else:
        if (adapter_exists or complete_checkpoints) and not reuse_existing:
            raise FileExistsError(
                f"Existing artifacts for {run_id} have no workflow session marker. "
                "Set REUSE_EXISTING_RUNS=True to adopt them explicitly."
            )
        session_path.parent.mkdir(parents=True, exist_ok=True)
        session_path.write_text(workflow_session_id + "\n", encoding="utf-8")
        volume.commit()
        same_session = True
    if adapter_exists:
        if reuse_existing or same_session:
            validate_adapter_dir(adapter_dir)
            print(f"Reusing completed adapter: {adapter_dir}", flush=True)
            return str(adapter_dir)
        raise FileExistsError(f"Adapter already exists for {run_id}: {adapter_dir}")
    if complete_checkpoints:
        print(f"Resuming {run_id} from {complete_checkpoints[-1].parent}", flush=True)

    warmstart_adapter = run_dir(BASELINE_RUN) / "adapter"
    validate_adapter_dir(warmstart_adapter)
    worker_config = {
        "run_id": run_id,
        "train_csv": str(Path(MOUNT_ROOT) / train_csv),
        "base_model": BASE_MODEL,
        "limit": 0,
        "max_steps": MAX_STEPS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "load_in_4bit": False,
        "lora_rank": 32,
        "lora_alpha": 32,
        "lora_dropout": 0.0,
        "warmstart_adapter": str(warmstart_adapter),
        "trainable_parameter_regex": None,
        "warmstart_delta_scale": 1.0,
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": PER_DEVICE_BATCH,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "max_grad_norm": MAX_GRAD_NORM,
        "lr_scheduler_type": LR_SCHEDULER,
        "warmup_steps": WARMUP_STEPS,
        "shuffle_train_dataset": True,
        "chat_template_prompts": True,
        "gradient_checkpointing": True,
        "seed": SEED,
        "checkpoint_steps": CHECKPOINT_STEPS,
        "checkpoint_keep": 2,
        "checkpoint_commit_callback": None,
    }
    config_path = Path("/tmp") / f"{run_id}_ddp_config.json"
    config_path.write_text(json.dumps(worker_config, sort_keys=True), encoding="utf-8")
    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nproc-per-node",
        str(TRAIN_WORLD_SIZE),
        "-m",
        "nemotron_reasoning.train_worker",
        "--config-json",
        str(config_path),
    ]
    print(
        json.dumps(
            {
                "event": "launch_training",
                "run_id": run_id,
                "gpu": TRAIN_GPU,
                "world_size": TRAIN_WORLD_SIZE,
                "effective_batch": EFFECTIVE_BATCH,
                "command": command,
            },
            sort_keys=True,
        ),
        flush=True,
    )
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    if process.stdout is None:
        raise RuntimeError("torchrun stdout pipe was not created")
    committed: set[str] = set()
    last_scan = time.monotonic()
    for line in process.stdout:
        print(line, end="", flush=True)
        now = time.monotonic()
        if "modal_checkpoint_complete.json" in line or now - last_scan >= 60:
            commit_complete_checkpoints(run_id, committed)
            last_scan = now
    return_code = process.wait()
    commit_complete_checkpoints(run_id, committed)
    volume.commit()
    if return_code != 0:
        raise RuntimeError(f"torchrun failed with exit code {return_code}")
    validate_adapter_dir(adapter_dir)
    return str(adapter_dir)


def upload_file(local: Path, remote: str) -> None:
    """Copy a local file onto the Modal volume."""
    with volume.batch_upload(force=True) as batch:
        batch.put_file(str(local), remote)


def download_file(remote: str, local: Path) -> Path:
    """Copy a file off the Modal volume to local scratch."""
    local.parent.mkdir(parents=True, exist_ok=True)
    with local.open("wb") as handle:
        for chunk in volume.read_file(remote):
            handle.write(chunk)
    return local

## Stage 1 - Baseline adapter

Download
[Kien's public Tinker adapter](https://www.kaggle.com/models/kienngx/nemotron-nano-30b-trained)
and place it on the Modal Volume at `runs/<BASELINE_RUN>/adapter/`. The adapter
is already in PEFT/vLLM-ready layout:

- `adapter_config.json`
- `adapter_model.safetensors`
- LoRA rank 32

The corresponding Kaggle
[submission notebook is here](https://www.kaggle.com/code/kienngx/nvidia-nemotron-trained-models-submission).

In [3]:
import kagglehub

kien_local = Path(kagglehub.model_download(KIEN_MODEL_HANDLE))
print("Downloaded Kien adapter to:", kien_local)
print("Validated Kien adapter:", validate_adapter_dir(kien_local))

# Stage the adapter files (config + weights, plus tokenizer if present).
adapter_files = ["adapter_config.json", "adapter_model.safetensors"]
with volume.batch_upload(force=True) as batch:
    for name in adapter_files:
        src = kien_local / name
        if not src.exists():
            raise FileNotFoundError(f"Kien adapter is missing {name} at {kien_local}")
        batch.put_file(str(src), f"runs/{BASELINE_RUN}/adapter/{name}")
    for extra in ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "chat_template.jinja"]:
        if (kien_local / extra).exists():
            batch.put_file(str(kien_local / extra), f"runs/{BASELINE_RUN}/adapter/{extra}")
print("Staged warm-start adapter at runs/%s/adapter/" % BASELINE_RUN)

# Upload your competition train.csv as well (used by every later stage).
if not LOCAL_TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"Competition train.csv not found at {LOCAL_TRAIN_CSV}. "
        "Download it from the Kaggle competition Data tab under your own access."
    )
from nemotron_reasoning.io_utils import read_csv_rows

source_rows = read_csv_rows(LOCAL_TRAIN_CSV)
if len(source_rows) != EXPECTED_FULL_ROWS:
    raise ValueError(f"Expected {EXPECTED_FULL_ROWS} competition rows, found {len(source_rows)}")
upload_file(LOCAL_TRAIN_CSV, "data/train.csv")
print("Uploaded train.csv to data/train.csv")

## Stage 2 - Baseline generation

Run the baseline adapter over all of `train.csv` with vLLM. The resulting
`train_full_vllm.csv` records each raw generation, the extracted answer, and
whether the answer was correct.

Prefix surgery uses incorrect rows as repair candidates. Correct rows from the
non-target task families are candidates for replay anchors.

In [4]:
with modal.enable_output(), app.run():
    baseline_summary = baseline_or_eval_infer.remote(
        run_id=BASELINE_RUN,
        csv_path="data/train.csv",
        output_name="train_full_vllm.csv",
        limit=0,
        tensor_parallel_size=1,
        use_adapter=True,
        reuse_existing=REUSE_EXISTING_RUNS,
    )
print("Baseline accuracy:", baseline_summary.get("accuracy"))

baseline_pred_local = download_file(
    f"runs/{BASELINE_RUN}/predictions/train_full_vllm.csv",
    WORK / "baseline_train_full_vllm.csv",
)
print("Downloaded baseline predictions to", baseline_pred_local)

## Stage 3 - Solver report

Run the deterministic solver over every `equation_symbol_cipher` puzzle. For
trace construction, the known training answer is used as an additional
constraint so the solver can recover a verified witness program. A separate
examples-only search records whether that program is unique, ambiguous, or
unknown.

This stage runs locally on CPU and writes an incremental solver report.

In [5]:
from nemotron_reasoning.symbol_cipher import run_report

solver_report_local = WORK / "solver_report.jsonl"
if solver_report_local.exists():
    if not REUSE_EXISTING_RUNS:
        raise FileExistsError(f"Solver report already exists: {solver_report_local}")
    solver_summary = {"total": len(solver_report_local.read_text(encoding="utf-8").splitlines())}
else:
    solver_summary = run_report(
        train_csv=LOCAL_TRAIN_CSV,
        out_jsonl=solver_report_local,
        workers=SOLVER_WORKERS,
        row_budget=SOLVER_ROW_BUDGET,
        uniqueness_budget=SOLVER_UNIQUENESS_BUDGET,
        progress_every=25,
    )
print("Solver summary:", {k: solver_summary[k] for k in ("total", "deduce", "guess") if k in solver_summary})

## Stage 4 - Build Arm A and Arm B

`build_corpora` applies the admission rules, creates the two trace styles, checks
every narrated calculation, and adds the same replay anchors to both arms.

- **Arm A:** keep the verified prefix from the baseline and train only the
  corrected continuation.
- **Arm B:** replace the baseline response with a clean trace written from the
  beginning.

The two training CSVs use the same row IDs. Their target-task trace format is
the controlled difference.

In [6]:
from nemotron_reasoning.corpus_build import build_corpora

trace_dir = WORK / SURGERY_RUN / "traces"
arm_a_local = WORK / ARM_A_RUN / "data" / "train.csv"
arm_b_local = WORK / ARM_B_RUN / "data" / "train.csv"

corpus_outputs = [trace_dir / "corpus_stats.json", arm_a_local, arm_b_local]
if any(path.exists() for path in corpus_outputs):
    if not REUSE_EXISTING_RUNS or not all(path.exists() for path in corpus_outputs):
        raise FileExistsError(f"Corpus artifacts already exist or are incomplete: {corpus_outputs}")
    corpus_stats = json.loads((trace_dir / "corpus_stats.json").read_text(encoding="utf-8"))
else:
    corpus_stats = build_corpora(
        train_csv=str(LOCAL_TRAIN_CSV),
        solver_report=str(solver_report_local),
        baseline_predictions=str(baseline_pred_local),
        trace_dir=str(trace_dir),
        arm_a_train=str(arm_a_local),
        arm_b_train=str(arm_b_local),
        seed=CORPUS_SEED,
        anchor_per_family=ANCHOR_PER_FAMILY,
    )
print("Inclusion counts:", corpus_stats["inclusion_counts"])
print("Arm A train rows:", corpus_stats["arm_a_train_rows"], "| Arm B train rows:", corpus_stats["arm_b_train_rows"])

# Upload both arm training CSVs to the volume for training.
upload_file(arm_a_local, f"runs/{ARM_A_RUN}/data/train.csv")
upload_file(arm_b_local, f"runs/{ARM_B_RUN}/data/train.csv")

## Stage 4b - Context-budget filter

Some Arm A rows have a long retained prefix. If the composed prompt and
continuation exceed the 8,192-token model limit, the notebook removes that row
from both arms. This keeps their training row sets identical and avoids
truncating the final answer.

In [7]:
with modal.enable_output(), app.run():
    filter_result = prefilter_overlength.remote(
        arm_a_csv=f"runs/{ARM_A_RUN}/data/train.csv",
        arm_b_csv=f"runs/{ARM_B_RUN}/data/train.csv",
        max_seq_length=MAX_SEQ_LENGTH,
    )
print("Dropped over-length ids:", filter_result["dropped_ids"])
print("Rows per arm after filter:", filter_result["arm_a_rows"])

## Stage 5 — Train both arms

Each arm warm-starts from the Kien adapter and trains with the identical
config (rank-32 LoRA with all Kien params trainable, completion-only loss,
chat-template prompts, `max_steps=223`, effective batch 16, LR 2e-5 cosine,
warmup 7, seed 12345, gradient checkpointing). Each arm uses two-process DDP
on two H200s. Both arms launch together, so this stage uses four H200s at peak.
Distributed reduction can move a few rows relative to the original single-GPU
run even though the optimizer configuration is preserved.

In [8]:
with modal.enable_output(), app.run():
    arm_a_call = train_arm_ddp.spawn(
        run_id=ARM_A_RUN,
        train_csv=f"runs/{ARM_A_RUN}/data/train.csv",
        workflow_session_id=WORKFLOW_SESSION_ID,
        reuse_existing=REUSE_EXISTING_RUNS,
    )
    arm_b_call = train_arm_ddp.spawn(
        run_id=ARM_B_RUN,
        train_csv=f"runs/{ARM_B_RUN}/data/train.csv",
        workflow_session_id=WORKFLOW_SESSION_ID,
        reuse_existing=REUSE_EXISTING_RUNS,
    )
    arm_a_adapter = arm_a_call.get()
    arm_b_adapter = arm_b_call.get()
    print("Arm A adapter:", arm_a_adapter)
    print("Arm B adapter:", arm_b_adapter)

## Stage 6 — Evaluate the full dataset and compare

Score both arms on all 9,500 rows, concurrently on one H200 each. The
canonical seed-12345 held-out view is then filtered from those deterministic
full predictions. The comparison reports overall, task-family, and
task-variant accuracy for both scopes.

In [9]:
from nemotron_reasoning.io_utils import read_csv_rows, write_csv_rows, write_json
from nemotron_reasoning.make_splits import split_train_valid

all_rows = read_csv_rows(LOCAL_TRAIN_CSV)
_, valid_rows = split_train_valid(all_rows, valid_fraction=VALID_FRACTION, seed=SPLIT_SEED)
full_ids = [row["id"] for row in all_rows]
valid_ids = [row["id"] for row in valid_rows]
if len(full_ids) != EXPECTED_FULL_ROWS or len(set(full_ids)) != EXPECTED_FULL_ROWS:
    raise AssertionError("The full source must contain exactly 9,500 unique ids")
if len(valid_ids) != EXPECTED_VALID_ROWS or len(set(valid_ids)) != EXPECTED_VALID_ROWS:
    raise AssertionError("The held-out split must contain exactly 1,899 unique ids")
print(f"Evaluation scopes: full={len(full_ids)}, held-out={len(valid_ids)}")

with modal.enable_output(), app.run():
    arm_a_eval_call = baseline_or_eval_infer.spawn(
        run_id=ARM_A_RUN,
        csv_path="data/train.csv",
        output_name="train_full_vllm.csv",
        use_adapter=True,
        reuse_existing=REUSE_EXISTING_RUNS,
    )
    arm_b_eval_call = baseline_or_eval_infer.spawn(
        run_id=ARM_B_RUN,
        csv_path="data/train.csv",
        output_name="train_full_vllm.csv",
        use_adapter=True,
        reuse_existing=REUSE_EXISTING_RUNS,
    )
    arm_a_eval = arm_a_eval_call.get()
    arm_b_eval = arm_b_eval_call.get()
print("Arm A full accuracy:", arm_a_eval.get("accuracy"))
print("Arm B full accuracy:", arm_b_eval.get("accuracy"))

arm_a_pred = download_file(
    f"runs/{ARM_A_RUN}/predictions/train_full_vllm.csv", WORK / "arm_a_train_full_vllm.csv"
)
arm_b_pred = download_file(
    f"runs/{ARM_B_RUN}/predictions/train_full_vllm.csv", WORK / "arm_b_train_full_vllm.csv"
)

In [10]:
from nemotron_reasoning.eval_analysis import build_comparison

payload, heldout_rows = build_comparison(
    baseline_rows=read_csv_rows(baseline_pred_local),
    arm_a_rows=read_csv_rows(arm_a_pred),
    arm_b_rows=read_csv_rows(arm_b_pred),
    full_ids=full_ids,
    valid_ids=valid_ids,
)

comparison_dir = WORK / SURGERY_RUN / "evaluation"
comparison_dir.mkdir(parents=True, exist_ok=True)
write_json(comparison_dir / "comparison.json", payload)


def flat_gate_rows(entries: list[dict], group_key: str, scope: str) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for entry in entries:
        row: dict[str, object] = {"scope": scope, group_key: entry[group_key]}
        for model in ("baseline", "arm_a", "arm_b"):
            cell = entry[model]
            row[f"{model}_correct"] = cell["correct"]
            row[f"{model}_total"] = cell["total"]
            row[f"{model}_accuracy"] = cell["accuracy"]
        rows.append(row)
    return rows


overall_rows: list[dict[str, object]] = []
for scope in ("full", "heldout"):
    for model, summary in payload[scope]["models"].items():
        overall_rows.append(
            {
                "scope": scope,
                "model": model,
                "correct": summary["correct"],
                "total": summary["scored_count"],
                "accuracy": summary["accuracy"],
            }
        )
write_csv_rows(
    comparison_dir / "comparison_overall.csv",
    overall_rows,
    ["scope", "model", "correct", "total", "accuracy"],
)

for scope in ("full", "heldout"):
    for group_key in ("task_family", "task_variant"):
        table = flat_gate_rows(payload[scope][f"gate_by_{group_key}"], group_key, scope)
        fields = ["scope", group_key]
        for model in ("baseline", "arm_a", "arm_b"):
            fields.extend([f"{model}_correct", f"{model}_total", f"{model}_accuracy"])
        write_csv_rows(comparison_dir / f"comparison_{scope}_by_{group_key}.csv", table, fields)


def display_scope(scope: str) -> None:
    print(f"\n{scope.upper()} OVERALL")
    for model, summary in payload[scope]["models"].items():
        print(f"  {model:9s} {summary['correct']}/{summary['scored_count']} = {summary['accuracy']:.4f}")
    for group_key in ("task_family", "task_variant"):
        print(f"\n{scope.upper()} BY {group_key.upper()}")
        for entry in payload[scope][f"gate_by_{group_key}"]:
            cells = []
            for model in ("baseline", "arm_a", "arm_b"):
                cell = entry[model]
                cells.append(f"{model}={cell['correct']}/{cell['total']} ({cell['accuracy']:.4f})")
            print(f"  {entry[group_key]}: " + ", ".join(cells))


display_scope("full")
display_scope("heldout")
print("\nSaved comparison artifacts to", comparison_dir)


FULL OVERALL
  baseline  8214/9500 = 0.8646
  arm_a     8081/9500 = 0.8506
  arm_b     7608/9500 = 0.8008

FULL BY TASK_FAMILY
  bit_manipulation: baseline=1261/1602 (0.7871), arm_a=1224/1602 (0.7640), arm_b=1233/1602 (0.7697)
  cipher_text: baseline=1565/1576 (0.9930), arm_a=1557/1576 (0.9879), arm_b=1553/1576 (0.9854)
  equation_transformation: baseline=629/1555 (0.4045), arm_a=544/1555 (0.3498), arm_b=64/1555 (0.0412)
  gravity: baseline=1593/1597 (0.9975), arm_a=1593/1597 (0.9975), arm_b=1591/1597 (0.9962)
  roman_numeral: baseline=1576/1576 (1.0000), arm_a=1576/1576 (1.0000), arm_b=1576/1576 (1.0000)
  unit_conversion: baseline=1590/1594 (0.9975), arm_a=1587/1594 (0.9956), arm_b=1591/1594 (0.9981)

FULL BY TASK_VARIANT
  bit_manipulation: baseline=1261/1602 (0.7871), arm_a=1224/1602 (0.7640), arm_b=1233/1602 (0.7697)
  cipher_text: baseline=1565/1576 (0.9930), arm_a=1557/1576 (0.9879), arm_b=1553/1576 (0.9854)
  equation_digit_ops: baseline=563/732 (0.7691), arm_a=534/732 (0.7295

## Notes

- **Compute:** training launches Arm A and Arm B concurrently with two H200s per
  arm, for a peak of four H200s. Evaluation launches one H200 per arm, for a
  peak of two H200s.
- **Persistence:** adapters, checkpoints, predictions, corpora, and comparison
  tables are stored on the configured Modal Volume and downloaded by the
  relevant notebook stages.
- **Write-up:** the motivation, trace examples, results, interpretation, and
  limitations are documented in [`WRITEUP.md`](../WRITEUP.md).

### References

- [Kien's public adapter](https://www.kaggle.com/models/kienngx/nemotron-nano-30b-trained)
- [Kien's submission notebook](https://www.kaggle.com/code/kienngx/nvidia-nemotron-trained-models-submission)
- [Official competition metric](https://www.kaggle.com/code/metric/nvidia-nemotron-metric)